Upload Library

In [ ]:
%pip install pandas numpy scipy scikit-learn tensorflow xgboost

In [1]:
import os
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import defaultdict
import warnings
import joblib
warnings.filterwarnings('ignore')
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost is available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not available")


XGBoost is available



SET parameters and DATA UTILITY FUNCTIONS

To set in DYNAMIC_PARAMS:
- M = n;
- 'NOISE_X' = if PR = 5 then "'NOISE_X': 5".

In [ ]:
# Seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(SEED)

# Dynamic parameters
DYNAMIC_PARAMS = {
    'M': 10,           # Thousands of samples (10 = 10,000)
    'TH': 20,          # Threshold (optional)
    'NOISE_X': 40,     # Percentage of randomness
}

# Paths
username = 'donatella.papa'  # <-- CHANGE THIS TO YOUR USERNAME
input_dir = rf"C:\Users\{username}\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step3"
models_dir = "./Output/output_keras"
reports_dir = "./Output/reports"

# Create directories
os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

# Column configuration
selected_columns = [
    'age',
    'municipality_residence',
    'civil_status',
    'gender',
    'occupation',
    'physical_activity',
    'genetic_predisposition',
    'diagnosis'
]

one_hot_columns = [
    'gender',
    'civil_status',
    'municipality_residence',
    'physical_activity',
    'genetic_predisposition',
    'age',
    'occupation'
]

# Model architecture
MLP_HIDDEN_LAYERS = [256, 128, 64, 32]
MLP_OUTPUT_SIZE = 4

# Training parameters
TRAINING_PARAMS = {
    'test_size': 0.2,
    'epochs': 30,
    'batch_size': 32,
    'learning_rate': 0.001,
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 0.001
}

# Model hyperparameters
RF_PARAMS = {
    'n_estimators': 200,
    'random_state': SEED,
    'class_weight': 'balanced',
    'n_jobs': -1
}
'''
# RANDOM FOREST REGOLARIZZATA
RF_PARAMS = {
    'n_estimators': 200,
    'random_state': SEED,
    'class_weight': 'balanced',
    'n_jobs': -1,
    # REGOLARIZZAZIONE: evita overfitting su dati rumorosi
    'max_depth': 10,              # Limita profondità degli alberi
    'min_samples_split': 20,       # Servono almeno 20 campioni per dividere un nodo
    'min_samples_leaf': 10,        # Ogni foglia deve avere almeno 10 campioni
    'max_features': 'sqrt'         # Usa solo sqrt(n_features) ad ogni split
}
'''
XGB_PARAMS = {
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.1,
    'objective': 'multi:softmax',
    'num_class': 4,
    'random_state': SEED,  
    'use_label_encoder': False,
    'eval_metric': 'mlogloss',
    'n_jobs': 1,
    'verbosity': 0,  
    'tree_method': 'exact' 
}

LR_PARAMS = {
    'max_iter': 2000,
    'random_state': SEED,
    'class_weight': 'balanced',
    'n_jobs': -1
}


def preprocess_dataset(file_path):
    """Preprocess a dataset by selecting columns and applying one-hot encoding."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File does not exist: {file_path}")
    
    df = pd.read_csv(file_path)
    df_filtered = df[selected_columns]
    df_encoded = pd.get_dummies(df_filtered, columns=one_hot_columns, drop_first=False)
    
    X = df_encoded.drop("diagnosis", axis=1)
    y = df_encoded["diagnosis"].values.ravel()
    
    return X, y

def align_datasets(synth_data, real_columns):
    """Align synthetic dataset to real dataset columns."""
    if isinstance(synth_data, dict):
        first_key = list(synth_data.keys())[0]
        synth_data = synth_data[first_key]
    
    synth_df = synth_data.copy()
    
    # Add missing columns
    for col in real_columns:
        if col not in synth_df.columns:
            synth_df[col] = 0
    
    # Remove extra columns
    for col in synth_df.columns:
        if col not in real_columns:
            synth_df = synth_df.drop(columns=[col])
    
    return synth_df[real_columns]

def align_to_mlp_features(mlp_model_path, X_data, dataset_name=""):
    """Align data to MLP expected features."""
    try:
        model = load_model(mlp_model_path)
        expected_features = model.input_shape[1]
        
        if isinstance(X_data, dict):
            X_data = X_data[list(X_data.keys())[0]]
        
        if not isinstance(X_data, pd.DataFrame):
            X_data = pd.DataFrame(X_data)
        
        if X_data.shape[1] == expected_features:
            return X_data, X_data.columns.tolist()
        
        if X_data.shape[1] > expected_features:
            aligned_cols = X_data.columns.tolist()[:expected_features]
        else:
            aligned_cols = X_data.columns.tolist()
            for i in range(expected_features - len(aligned_cols)):
                aligned_cols.append(f"extra_col_{i}")
        
        X_aligned = X_data[aligned_cols].copy()
        print(f"{dataset_name}: {X_data.shape[1]} -> {X_aligned.shape[1]} features")
        
        return X_aligned, aligned_cols
        
    except Exception as e:
        print(f"Alignment error {dataset_name}: {e}")
        return X_data, X_data.columns.tolist() if hasattr(X_data, 'columns') else []

def align_df_to_mlp_features(df, mlp_features):
    """Align DataFrame to MLP features."""
    if not isinstance(df, pd.DataFrame):
        if hasattr(df, 'columns'):
            df = pd.DataFrame(df, columns=df.columns)
        else:
            if len(mlp_features) == df.shape[1]:
                return pd.DataFrame(df, columns=mlp_features)
            else:
                return df
    
    df_aligned = df.copy()
    
    # Add missing columns
    for col in mlp_features:
        if col not in df_aligned.columns:
            df_aligned[col] = 0
    
    # Remove extra columns
    extra_cols = [col for col in df_aligned.columns if col not in mlp_features]
    if extra_cols:
        df_aligned = df_aligned.drop(columns=extra_cols)
    
    return df_aligned[mlp_features]

def remove_X2_features(dataset):
    """Remove X2 features (physical_activity and genetic_predisposition)."""
    if isinstance(dataset, dict):
        dataset = dataset[list(dataset.keys())[0]]
    
    dataset_mod = dataset.copy()
    
    # Find X2 columns
    pa_cols = [col for col in dataset_mod.columns if col.startswith('physical_activity_')]
    gp_cols = [col for col in dataset_mod.columns if col.startswith('genetic_predisposition_')]
    
    # Set to 0
    for col in pa_cols + gp_cols:
        if col in dataset_mod.columns:
            dataset_mod[col] = 0
    
    # Set Missing to 1
    if 'physical_activity_Missing' in dataset_mod.columns:
        dataset_mod['physical_activity_Missing'] = 1
    if 'genetic_predisposition_Missing' in dataset_mod.columns:
        dataset_mod['genetic_predisposition_Missing'] = 1
    
    return dataset_mod

def apply_sampling(X, y, M_k):
    """Apply sampling to reduce dataset size."""
    try:
        desired = int(M_k) * 1000
        if 0 < desired < len(y):
            X_sample, _, y_sample, _ = train_test_split(
                X, y, train_size=desired, random_state=SEED, stratify=y
            )
            return X_sample, y_sample
    except Exception as e:
        print(f"Sampling error: {e}")
    return X, y

MODEL UTILITY FUNCTIONS

In [28]:

def get_model_filename(cfg, model_type, model_suffix=""):
    """Generate filename for different model types."""
    name = cfg["name"]
    synth_type = cfg.get("synth_type", "")
    synth_version = cfg.get("synth_version", "")
    
    # Determine base pattern
    if cfg["type"] == "real":
        base_name = "REAL"
    else:
        if synth_type and synth_version:
            base_name = f"SYNTHETIC_{synth_type.upper()}_{synth_version}"
        elif synth_type:
            base_name = f"SYNTHETIC_{synth_type.upper()}"
        else:
            base_name = "SYNTHETIC"
    
    # Add noise if present
    noise_suffix = f"_NOISE{DYNAMIC_PARAMS['NOISE_X']}" if "random" in name.lower() else ""
    
    # Build filename
    if model_type == "mlp":
        return f"mlp_{base_name}_M{DYNAMIC_PARAMS['M']}_TH{DYNAMIC_PARAMS['TH']}{noise_suffix}{model_suffix}.keras"
    elif model_type == "lr":
        return f"lr_{base_name}_M{DYNAMIC_PARAMS['M']}_TH{DYNAMIC_PARAMS['TH']}{noise_suffix}{model_suffix}.pkl"
    elif model_type == "rf":
        return f"rf_{base_name}_M{DYNAMIC_PARAMS['M']}_TH{DYNAMIC_PARAMS['TH']}{noise_suffix}{model_suffix}.pkl"
    elif model_type == "xgb":
        return f"xgb_{base_name}_M{DYNAMIC_PARAMS['M']}_TH{DYNAMIC_PARAMS['TH']}{noise_suffix}{model_suffix}.pkl"
    else:
        return f"{model_type}_{base_name}_M{DYNAMIC_PARAMS['M']}_TH{DYNAMIC_PARAMS['TH']}{noise_suffix}{model_suffix}.pkl"

def create_mlp_model_4hl(input_dim):
    """Create MLP with 4 hidden layers."""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(MLP_HIDDEN_LAYERS[0], activation='relu'),
        Dense(MLP_HIDDEN_LAYERS[1], activation='relu'),
        Dense(MLP_HIDDEN_LAYERS[2], activation='relu'),
        Dense(MLP_HIDDEN_LAYERS[3], activation='relu'),
        Dense(MLP_OUTPUT_SIZE, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=TRAINING_PARAMS['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy', 'precision', 'recall', 'auc']
    )
    
    return model

def train_mlp_model(cfg):
    """Train MLP model for a single configuration."""
    name = cfg["name"]
    mlp_path = cfg["mlp_path"]
    
    print(f"\n[{name}] Training MLP...")
    
    if "X_train_aligned" not in cfg or "y_train" not in cfg:
        print("  ✗ Skipped - incomplete configuration")
        return False
    
    if os.path.exists(mlp_path):
        print("  ✓ Model already exists, skipped")
        return False
    
    try:
        X_data = cfg["X_train_aligned"].astype(np.float32)
        y_data = to_categorical(cfg["y_train"] - 1, num_classes=4)
        
        X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
            X_data, y_data, 
            test_size=TRAINING_PARAMS['test_size'], 
            random_state=SEED, 
            stratify=cfg["y_train"]
        )
        
        model = create_mlp_model_4hl(X_data.shape[1])
        
        callbacks = [
            ModelCheckpoint(mlp_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0),
            EarlyStopping(monitor='val_accuracy', patience=TRAINING_PARAMS['early_stopping_patience'],
                         min_delta=TRAINING_PARAMS['early_stopping_min_delta'], 
                         mode='max', restore_best_weights=True, verbose=0)
        ]
        
        history = model.fit(
            X_train_split, y_train_split,
            epochs=TRAINING_PARAMS['epochs'],
            batch_size=TRAINING_PARAMS['batch_size'],
            validation_data=(X_val_split, y_val_split),
            verbose=0,
            callbacks=callbacks
        )
        
        model.save(mlp_path.replace('.keras', '_final.keras'))
        
        history_df = pd.DataFrame(history.history)
        history_df['epoch'] = range(1, len(history_df) + 1)
        history_df.to_csv(mlp_path.replace('.keras', '_history.csv'), index=False, sep=';')
        
        val_acc = max(history.history['val_accuracy'])
        print(f"  ✓ Completed - Val accuracy: {val_acc:.4f}")
        return True
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return False

def train_lr_model(cfg):
    """Train Logistic Regression model."""
    try:
        model = LogisticRegression(**LR_PARAMS)
        model.fit(cfg["X_train_aligned"], cfg["y_train"])
        joblib.dump(model, cfg["lr_path"])
        print(f"  ✓ Logistic Regression saved")
        return True
    except Exception as e:
        print(f"  ✗ LR error: {e}")
        return False

def train_rf_model(cfg):
    """Train Random Forest model."""
    try:
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(cfg["X_train_aligned"], cfg["y_train"])
        joblib.dump(model, cfg["rf_path"])
        print(f"  ✓ Random Forest saved")
        return True
    except Exception as e:
        print(f"  ✗ RF error: {e}")
        return False

def train_xgb_model(cfg):
    """Train XGBoost model."""
    if not XGBOOST_AVAILABLE:
        return False
    try:
        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(cfg["X_train_aligned"], cfg["y_train"] - 1)
        joblib.dump(model, cfg["xgb_path"])
        print(f"  ✓ XGBoost saved")
        return True
    except Exception as e:
        print(f"  ✗ XGB error: {e}")
        return False

def calculate_mlp_params(input_dim):
    """Calculate MLP parameters."""
    layers = [input_dim] + MLP_HIDDEN_LAYERS + [MLP_OUTPUT_SIZE]
    total_params = 0
    for i in range(len(layers)-1):
        total_params += (layers[i] * layers[i+1]) + layers[i+1]
    return total_params

ANALYSIS FUNCTIONS

In [29]:
def random_mode(x):
    """Calculate mode with random selection in case of tie."""
    counts = x.value_counts()
    max_count = counts.max()
    modes = counts[counts == max_count].index.tolist()
    return np.random.choice(modes)

def create_benchmark_model(X_train, y_train):
    """Create baseline model based on age."""
    try:
        if not isinstance(X_train, pd.DataFrame):
            X_train = pd.DataFrame(X_train)
        
        age_cols = X_train.filter(regex='^age_')
        train_df = pd.DataFrame({
            'age': age_cols.idxmax(axis=1).str.replace('age_', '').astype(int),
            'diagnosis': y_train
        })
        
        age_mode_map = train_df.groupby('age')['diagnosis'].agg(random_mode).to_dict()
        global_mode = train_df['diagnosis'].mode()[0] if len(train_df['diagnosis'].mode()) > 0 else 1
        
        return age_mode_map, global_mode
    except Exception as e:
        print(f"Benchmark error: {e}")
        return {}, 1

def benchmark_predict(X_test, age_mode_map, global_mode):
    """Prediction with baseline model."""
    try:
        if not isinstance(X_test, pd.DataFrame):
            X_test = pd.DataFrame(X_test)
        
        age_cols = X_test.filter(regex='^age_')
        test_ages = age_cols.idxmax(axis=1).str.replace('age_', '').astype(int)
        
        preds = [age_mode_map.get(age, global_mode) for age in test_ages]
        return np.array(preds)
    except Exception as e:
        print(f"Benchmark prediction error: {e}")
        return np.ones(len(X_test))

def evaluate_configuration(cfg, eval_type, use_nox2=False):
    """Evaluate all models for a configuration."""
    name = cfg["name"]
    results = {
        "Benchmark": np.nan,
        "MLP (4 hl)": np.nan,
        "Logistic Regression": np.nan,
        "Random Forest": np.nan
    }
    if XGBOOST_AVAILABLE:
        results["XGBoost"] = np.nan
    
    # Prepare evaluation data
    try:
        if use_nox2:
            if "Training" in eval_type:
                real_key = f"RY{DYNAMIC_PARAMS['NOISE_X']}" if "random" in name.lower() else "RN0"
                X_eval = remove_X2_features(align_df_to_mlp_features(X_train_real[real_key], cfg["mlp_features"]))
                y_eval = y_train_real[real_key]
            else:
                X_eval = remove_X2_features(cfg["X_test_real_aligned"])
                y_eval = cfg["y_test_real"]
        else:
            if "Training" in eval_type:
                real_key = f"RY{DYNAMIC_PARAMS['NOISE_X']}" if "random" in name.lower() else "RN0"
                X_eval = align_df_to_mlp_features(X_train_real[real_key], cfg["mlp_features"])
                y_eval = y_train_real[real_key]
            else:
                X_eval = cfg["X_test_real_aligned"]
                y_eval = cfg["y_test_real"]
    except Exception as e:
        print(f"Data prep error: {e}")
        return results
    
    # Benchmark
    try:
        age_mode_map, global_mode = create_benchmark_model(cfg["X_train_aligned"], cfg["y_train"])
        results["Benchmark"] = accuracy_score(y_eval, benchmark_predict(X_eval, age_mode_map, global_mode))
    except:
        pass
    
    # MLP
    if os.path.exists(cfg["mlp_path"]):
        try:
            model = load_model(cfg["mlp_path"])
            y_pred = np.argmax(model.predict(X_eval.values.astype(np.float32), verbose=0), axis=1)
            results["MLP (4 hl)"] = accuracy_score(y_eval - 1, y_pred)
        except Exception as e:
            print(f"MLP error: {e}")
    
    # Logistic Regression
    if os.path.exists(cfg["lr_path"]):
        try:
            model = joblib.load(cfg["lr_path"])
            results["Logistic Regression"] = accuracy_score(y_eval, model.predict(X_eval))
        except:
            pass
    
    # Random Forest
    if os.path.exists(cfg["rf_path"]):
        try:
            model = joblib.load(cfg["rf_path"])
            results["Random Forest"] = accuracy_score(y_eval, model.predict(X_eval))
        except:
            pass
    
    # XGBoost
    if XGBOOST_AVAILABLE and os.path.exists(cfg["xgb_path"]):
        try:
            model = joblib.load(cfg["xgb_path"])
            results["XGBoost"] = accuracy_score(y_eval - 1, model.predict(X_eval))
        except:
            pass
    
    return results

LOAD REAL DATASETS

In [30]:
print("="*60)
print("LOADING REAL DATASETS")
print("="*60)

# Define paths for Step1 and Step2
step1_dir = rf"C:\Users\{username}\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\output"
step2_dir = rf"C:\Users\{username}\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step2\output"

print(f"Step1: {step1_dir}")
print(f"Step2: {step2_dir}")

# Real datasets configuration
real_configs = [
    {"key": "RN0", "R": "N", "PR": 0},
    {"key": "RY40", "R": "Y", "PR": 40}
]

X_train_real, X_test_real = {}, {}
y_train_real, y_test_real = {}, {}

for rc in real_configs:
    file_path = os.path.join(
        step1_dir,
        f'real_data_datasetM10_TH20_R{rc["R"]}_PR{rc["PR"]}_4CAT_MISS_X2.csv'
    )
    
    if os.path.exists(file_path):
        X, y = preprocess_dataset(file_path)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=SEED
        )
        X_train_real[rc["key"]] = X_train
        X_test_real[rc["key"]] = X_test
        y_train_real[rc["key"]] = y_train
        y_test_real[rc["key"]] = y_test
        print(f"✓ Loaded {rc['key']}: {X.shape}")
    else:
        print(f"✗ File not found: {file_path}")

LOADING REAL DATASETS
Step1: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\output
Step2: C:\Users\donatella.papa\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step2\output
✓ Loaded RN0: (10000, 217)
✓ Loaded RY40: (10000, 217)


LOAD SYNTHETIC DATASETS

In [31]:
print("\n" + "="*60)
print("LOADING SYNTHETIC DATASETS")
print("="*60)

synthetic_configs = {
    "SN0_tvae": {"R": "N", "PR": 0, "tipo": "tvae"},
    "SY40_tvae": {"R": "Y", "PR": 40, "tipo": "tvae"},
    "SN0_ctgan": {"R": "N", "PR": 0, "tipo": "ctgan"},
    "SY40_ctgan": {"R": "Y", "PR": 40, "tipo": "ctgan"},
    "SN0_xgboost_O0": {"R": "N", "PR": 0, "tipo": "XGBoost_O0"},
    "SY40_xgboost_O0": {"R": "Y", "PR": 40, "tipo": "XGBoost_O0"}
}

X_train_synth, X_test_synth = {}, {}
y_train_synth, y_test_synth = {}, {}

for key, params in synthetic_configs.items():
    file_path = os.path.join(
        step2_dir,
        f'synthetic_data_datasetM10_TH20_R{params["R"]}_PR{params["PR"]}_4CAT_MISS_X2_{params["tipo"]}.csv'
    )
    
    if os.path.exists(file_path):
        X, y = preprocess_dataset(file_path)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=SEED
        )
        X_train_synth[key] = X_train
        X_test_synth[key] = X_test
        y_train_synth[key] = y_train
        y_test_synth[key] = y_test
        print(f"✓ Loaded {key}: {X.shape}")
    else:
        print(f"✗ File not found: {key}")


print("\n" + "="*60)
print("ALIGNING SYNTHETIC DATASETS")
print("="*60)

# Usa RN0 come riferimento per le colonne
real_columns = X_train_real["RN0"].columns.tolist()
print(f"Reference columns from RN0: {len(real_columns)}")

X_train_synth_aligned = {}
for key in X_train_synth.keys():
    X_train_synth_aligned[key] = align_datasets(X_train_synth[key], real_columns)
    print(f"✓ {key}: {X_train_synth[key].shape[1]} -> {X_train_synth_aligned[key].shape[1]} columns")


LOADING SYNTHETIC DATASETS
✓ Loaded SN0_tvae: (10000, 189)
✓ Loaded SY40_tvae: (10000, 197)
✓ Loaded SN0_ctgan: (10000, 217)
✓ Loaded SY40_ctgan: (10000, 217)
✓ Loaded SN0_xgboost_O0: (10000, 216)
✓ Loaded SY40_xgboost_O0: (10000, 216)

ALIGNING SYNTHETIC DATASETS
Reference columns from RN0: 217
✓ SN0_tvae: 189 -> 217 columns
✓ SY40_tvae: 197 -> 217 columns
✓ SN0_ctgan: 217 -> 217 columns
✓ SY40_ctgan: 217 -> 217 columns
✓ SN0_xgboost_O0: 216 -> 217 columns
✓ SY40_xgboost_O0: 216 -> 217 columns


Model Configuration

In [32]:
print("\n" + "="*60)
print("CREATING MODEL CONFIGURATIONS")
print("="*60)

configs = []  

# Real configurations
real_configs = [
    {"name": "Real", "train_key": "RN0", "test_key": "RN0"},
    {"name": f"Real - {DYNAMIC_PARAMS['NOISE_X']}% random", 
     "train_key": f"RY{DYNAMIC_PARAMS['NOISE_X']}", 
     "test_key": f"RY{DYNAMIC_PARAMS['NOISE_X']}"}
]

for rc in real_configs:
    cfg = {
        "name": rc["name"],
        "type": "real",
        "train_key": rc["train_key"],
        "test_key": rc["test_key"],
        "X_train": X_train_real[rc["train_key"]],
        "y_train": y_train_real[rc["train_key"]],
        "X_test_real": X_test_real[rc["test_key"]],
        "y_test_real": y_test_real[rc["test_key"]],
    }
    # Add model paths
    for mt in ["mlp", "lr", "rf", "xgb"]:
        cfg[f"{mt}_path"] = os.path.join(models_dir, get_model_filename(cfg, mt))
    configs.append(cfg)
    print(f"✓ Added real: {rc['name']}")

# Synthetic configurations
for synth_key in sorted(X_train_synth_aligned.keys()):
    parts = synth_key.split('_')
    
    if len(parts) >= 2:
        base_config = parts[0]
        synth_type = parts[1]
        synth_version = parts[2] if len(parts) > 2 else ""
        R = base_config[1]
        
        model_name = f"Synthetic ({synth_type.upper()}"
        if synth_version:
            model_name += f" {synth_version}"
        model_name += ")"
        if R == "Y":
            model_name += f" - {DYNAMIC_PARAMS['NOISE_X']}% random"
        
        test_key = "RN0" if R == "N" else f"RY{DYNAMIC_PARAMS['NOISE_X']}"
        
        cfg = {
            "name": model_name,
            "type": "synthetic",
            "synth_type": synth_type,
            "synth_version": synth_version,
            "train_key": synth_key,
            "test_key": test_key,
            "X_train": X_train_synth_aligned[synth_key],
            "y_train": y_train_synth[synth_key],
            "X_test_real": X_test_real[test_key],
            "y_test_real": y_test_real[test_key],
        }
        # Add model paths
        for mt in ["mlp", "lr", "rf", "xgb"]:
            cfg[f"{mt}_path"] = os.path.join(models_dir, get_model_filename(cfg, mt))
        
        configs.append(cfg)
        print(f"✓ Added synthetic: {model_name}")

print(f"\nTotal configurations: {len(configs)}")


CREATING MODEL CONFIGURATIONS
✓ Added real: Real
✓ Added real: Real - 40% random
✓ Added synthetic: Synthetic (CTGAN)
✓ Added synthetic: Synthetic (TVAE)
✓ Added synthetic: Synthetic (XGBOOST O0)
✓ Added synthetic: Synthetic (CTGAN) - 40% random
✓ Added synthetic: Synthetic (TVAE) - 40% random
✓ Added synthetic: Synthetic (XGBOOST O0) - 40% random

Total configurations: 8


FEATURE ALIGNMENT FOR MLP AND sAMPLYNG

In [33]:
print("\n" + "="*60)
print("FEATURE ALIGNMENT FOR MLP")
print("="*60)

for cfg in configs:
    if os.path.exists(cfg["mlp_path"]):
        cfg["X_train_aligned"], cfg["mlp_features"] = align_to_mlp_features(
            cfg["mlp_path"], cfg["X_train"], f"Train {cfg['name']}"
        )
        cfg["X_test_real_aligned"], _ = align_to_mlp_features(
            cfg["mlp_path"], cfg["X_test_real"], f"Test {cfg['name']}"
        )
        print(f"✓ {cfg['name']}: {cfg['X_train_aligned'].shape[1]} features (aligned to existing MLP)")
    else:
        cfg["X_train_aligned"] = cfg["X_train"]
        cfg["X_test_real_aligned"] = cfg["X_test_real"]
        cfg["mlp_features"] = cfg["X_train"].columns.tolist() if hasattr(cfg["X_train"], 'columns') else [f"f{i}" for i in range(cfg["X_train"].shape[1])]
        print(f"✓ {cfg['name']}: {len(cfg['mlp_features'])} features (new model)")
        
print("\n" + "="*60)
print("APPLYING SAMPLING")
print("="*60)

for cfg in configs:
    cfg["X_train_aligned"], cfg["y_train"] = apply_sampling(
        cfg["X_train_aligned"], cfg["y_train"], DYNAMIC_PARAMS['M']
    )
    print(f"✓ {cfg['name']}: {cfg['X_train_aligned'].shape[0]} samples")


FEATURE ALIGNMENT FOR MLP
✓ Real: 217 features (aligned to existing MLP)
✓ Real - 40% random: 217 features (aligned to existing MLP)
✓ Synthetic (CTGAN): 217 features (aligned to existing MLP)
✓ Synthetic (TVAE): 217 features (aligned to existing MLP)
✓ Synthetic (XGBOOST O0): 217 features (aligned to existing MLP)
✓ Synthetic (CTGAN) - 40% random: 217 features (aligned to existing MLP)
✓ Synthetic (TVAE) - 40% random: 217 features (aligned to existing MLP)
✓ Synthetic (XGBOOST O0) - 40% random: 217 features (aligned to existing MLP)

APPLYING SAMPLING
✓ Real: 8000 samples
✓ Real - 40% random: 8000 samples
✓ Synthetic (CTGAN): 8000 samples
✓ Synthetic (TVAE): 8000 samples
✓ Synthetic (XGBOOST O0): 8000 samples
✓ Synthetic (CTGAN) - 40% random: 8000 samples
✓ Synthetic (TVAE) - 40% random: 8000 samples
✓ Synthetic (XGBOOST O0) - 40% random: 8000 samples


TRAIN MODELS (OPTIONAL IF MODEL ARE READY IN FOLDER)

In [39]:
train_models = input("\nTrain missing models? (y/n): ").lower() == 'y'

if train_models:
    print("\n" + "="*60)
    print("TRAINING MODELS")
    print("="*60)
    
    for model_type in ["mlp", "lr", "rf", "xgb"]:
        if model_type == "xgb" and not XGBOOST_AVAILABLE:
            continue
            
        print(f"\n{'-'*50}")
        print(f"TRAINING {model_type.upper()} MODELS")
        print(f"{'-'*50}")
        
        trained = 0
        skipped = 0
        
        for cfg in configs:
            if os.path.exists(cfg[f"{model_type}_path"]):
                skipped += 1
                continue
            
            if model_type == "mlp":
                success = train_mlp_model(cfg)
            elif model_type == "lr":
                success = train_lr_model(cfg)
            elif model_type == "rf":
                success = train_rf_model(cfg)
            elif model_type == "xgb":
                success = train_xgb_model(cfg)
            
            if success:
                trained += 1
        
        print(f"\n{model_type.upper()} Summary: Trained {trained}, Skipped {skipped}")


TRAINING MODELS

--------------------------------------------------
TRAINING MLP MODELS
--------------------------------------------------

MLP Summary: Trained 0, Skipped 8

--------------------------------------------------
TRAINING LR MODELS
--------------------------------------------------

LR Summary: Trained 0, Skipped 8

--------------------------------------------------
TRAINING RF MODELS
--------------------------------------------------

RF Summary: Trained 0, Skipped 8

--------------------------------------------------
TRAINING XGB MODELS
--------------------------------------------------

XGB Summary: Trained 0, Skipped 8


Da cancellare: verifica della random forest

In [ ]:
print("\n" + "="*60)
print("VERIFICA PARAMETRI RANDOM FOREST DOPO REGOLARIZZAZIONE")
print("="*60)

for cfg in configs:
    if "Real - 40% random" in cfg["name"]:
        print(f"\n Analisi {cfg['name']}:")
        
        rf_path = cfg["rf_path"]
        if os.path.exists(rf_path):
            rf = joblib.load(rf_path)
            
            # Stampa i parametri effettivi del modello
            print(f"\n PARAMETRI RANDOM FOREST:")
            print(f"  - max_depth: {rf.max_depth}")
            print(f"  - min_samples_split: {rf.min_samples_split}")
            print(f"  - min_samples_leaf: {rf.min_samples_leaf}")
            print(f"  - max_features: {rf.max_features}")
            print(f"  - n_estimators: {rf.n_estimators}")
            
            # Verifica profondità effettiva degli alberi
            depths = [tree.tree_.max_depth for tree in rf.estimators_]
            print(f"  - Profondità media alberi: {np.mean(depths):.1f}")
            print(f"  - Profondità max alberi: {max(depths)}")
            print(f"  - Profondità min alberi: {min(depths)}")
            
            # Verifica dimensione foglie
            leaf_samples = [tree.tree_.n_node_samples[tree.tree_.children_left == -1].sum() for tree in rf.estimators_]
            print(f"  - Media campioni per foglia: {np.mean(leaf_samples):.1f}")


VERIFICA PARAMETRI RANDOM FOREST DOPO REGOLARIZZAZIONE

🔍 Analisi Real - 40% random:

📊 PARAMETRI RANDOM FOREST:
  - max_depth: None
  - min_samples_split: 2
  - min_samples_leaf: 1
  - max_features: sqrt
  - n_estimators: 200
  - Profondità media alberi: 90.0
  - Profondità max alberi: 115
  - Profondità min alberi: 71
  - Media campioni per foglia: 5057.9


c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
print("\n" + "="*60)
print("PERFORMANCE RANDOM FOREST REGOLARIZZATA")
print("="*60)

for cfg in configs:
    if "Real - 40% random" in cfg["name"]:
        print(f"\n {cfg['name']}:")
        
        rf = joblib.load(cfg["rf_path"])
        
        # Training set
        train_pred = rf.predict(cfg["X_train_aligned"])
        train_acc = accuracy_score(cfg["y_train"], train_pred)
        
        # Test set
        test_pred = rf.predict(cfg["X_test_real_aligned"])
        test_acc = accuracy_score(cfg["y_test_real"], test_pred)
        
        print(f" TRAIN accuracy: {train_acc:.4f}")
        print(f"  TEST accuracy:  {test_acc:.4f}")
        print(f"  GAP: {train_acc - test_acc:.4f}")
        
        # Confronto con gli altri modelli
        print(f"\n CONFRONTO CON ALTRI MODELLI (TEST):")
        
        # Logistic Regression
        if os.path.exists(cfg["lr_path"]):
            lr = joblib.load(cfg["lr_path"])
            lr_test_acc = accuracy_score(cfg["y_test_real"], lr.predict(cfg["X_test_real_aligned"]))
            print(f"     LR: {lr_test_acc:.4f}")
        
        # MLP
        if os.path.exists(cfg["mlp_path"]):
            mlp = load_model(cfg["mlp_path"])
            X_test_mlp = cfg["X_test_real_aligned"].values.astype(np.float32)
            mlp_pred = np.argmax(mlp.predict(X_test_mlp, verbose=0), axis=1) + 1
            mlp_test_acc = accuracy_score(cfg["y_test_real"], mlp_pred)
            print(f"     MLP: {mlp_test_acc:.4f}")
        
        # XGB
        if XGBOOST_AVAILABLE and os.path.exists(cfg["xgb_path"]):
            xgb_model = joblib.load(cfg["xgb_path"])
            xgb_pred = xgb_model.predict(cfg["X_test_real_aligned"]) + 1
            xgb_test_acc = accuracy_score(cfg["y_test_real"], xgb_pred)
            print(f"     XGB: {xgb_test_acc:.4f}")


PERFORMANCE RANDOM FOREST REGOLARIZZATA

🔍 Real - 40% random:


c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.w

  📊 TRAIN accuracy: 0.9948
  📊 TEST accuracy:  0.5350
  📊 GAP: 0.4597

  📊 CONFRONTO CON ALTRI MODELLI (TEST):
     LR: 0.5425
     MLP: 0.5460
     XGB: 0.5455


EVALUATE ALL MODELS

In [21]:
print("\n" + "="*60)
print("EVALUATING ALL MODELS")
print("="*60)

evaluations = [
    {"name": "Test", "nox2": False},
    {"name": "Training", "nox2": False},
    {"name": "Test - without X2", "nox2": True},
    {"name": "Training - without X2", "nox2": True}
]

all_results = []

for eval_config in evaluations:
    print(f"\n{'-'*50}")
    print(f"EVALUATION: {eval_config['name']}")
    print(f"{'-'*50}")
    
    for cfg in configs:
        results = evaluate_configuration(cfg, eval_config['name'], eval_config['nox2'])
        
        result_record = {
            "Model": cfg["name"],
            "Evaluation": eval_config['name'],
            "Benchmark": results["Benchmark"],
            "MLP": results["MLP (4 hl)"],
            "LR": results["Logistic Regression"],
            "RF": results["Random Forest"]
        }
        if XGBOOST_AVAILABLE:
            result_record["XGB"] = results["XGBoost"]
        
        all_results.append(result_record)
        
        # Print summary line
        scores = [f"{k}:{v:.3f}" for k, v in results.items() if not np.isnan(v)]
        print(f"  {cfg['name'][:30]:<30} {', '.join(scores)}")


EVALUATING ALL MODELS

--------------------------------------------------
EVALUATION: Test
--------------------------------------------------
  Real                           Benchmark:0.356, MLP (4 hl):1.000, Logistic Regression:0.997, Random Forest:0.992, XGBoost:0.993
  Real - 40% random              Benchmark:0.292, MLP (4 hl):0.546, Logistic Regression:0.542, Random Forest:0.535, XGBoost:0.545
  Synthetic (CTGAN)              Benchmark:0.306, MLP (4 hl):0.853, Logistic Regression:0.854, Random Forest:0.834, XGBoost:0.833
  Synthetic (TVAE)               Benchmark:0.285, MLP (4 hl):0.657, Logistic Regression:0.683, Random Forest:0.653, XGBoost:0.659
  Synthetic (XGBOOST O0)         Benchmark:0.354, MLP (4 hl):0.964, Logistic Regression:0.982, Random Forest:0.947, XGBoost:0.947
  Synthetic (CTGAN) - 40% random Benchmark:0.300, MLP (4 hl):0.520, Logistic Regression:0.513, Random Forest:0.481, XGBoost:0.513
  Synthetic (TVAE) - 40% random  Benchmark:0.276, MLP (4 hl):0.387, Logistic 

c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning:

  Real - 40% random              Benchmark:0.319, MLP (4 hl):0.333, Logistic Regression:0.307, Random Forest:0.424, XGBoost:0.339


c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning:

  Synthetic (CTGAN)              Benchmark:0.333, MLP (4 hl):0.280, Logistic Regression:0.265, Random Forest:0.270, XGBoost:0.239


...

c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning:

  Synthetic (TVAE) - 40% random  Benchmark:0.261, MLP (4 hl):0.274, Logistic Regression:0.282, Random Forest:0.277, XGBoost:0.266


c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning:

  Synthetic (XGBOOST O0) - 40% r Benchmark:0.306, MLP (4 hl):0.298, Logistic Regression:0.302, Random Forest:0.283, XGBoost:0.300


c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\donatella.papa\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.w

SAVE RESULTS AND SUMMARY

In [ ]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Save to CSV
results_csv = os.path.join(reports_dir, "complete_results.csv")
results_df.to_csv(results_csv, index=False, sep=';')
print(f"Results saved to: {results_csv}")

# Create summary
print("\nACCURACY SUMMARY (Test only):")
test_results = results_df[results_df['Evaluation'] == 'Test']
summary = test_results.groupby('Model').agg({
    'MLP': 'mean',
    'LR': 'mean',
    'RF': 'mean',
    'Benchmark': 'mean'
}).round(3)

if XGBOOST_AVAILABLE:
    summary['XGB'] = test_results.groupby('Model')['XGB'].mean().round(3)

print(summary.to_string())

# Save summary
summary_csv = os.path.join(reports_dir, "accuracy_summary.csv")
summary.to_csv(summary_csv, sep=';')
print(f"\nSummary saved to: {summary_csv}")